# Notebook 7 — Inferência de Desempenho: Significância Estatística da Comparação

**TCC — Pedro Augusto Pinheiro Reis · Ciências Contábeis · Universidade Federal de Goiás (UFG)**
*Moderna Teoria das Carteiras no Mercado de Ações Brasileiro: Comparação entre Otimizadores e Inputs*
Orientador: Prof. Dr. Moisés Ferreira da Cunha

| Metadado | Valor |
| :--- | :--- |
| Autor | Pedro Augusto Pinheiro Reis · *(e-mail a preencher)* |
| Instituição | FACE/UFG — Ciências Contábeis |
| Data da análise | *(preencher)* |
| Insumo | `data/Estrategias/strategy_returns.csv` (NB06, 15 estratégias + IBOV), `rf_diario` (NB02) |
| Saídas | `inferencia_diagnosticos.csv`, `inferencia_capm.csv`, `inferencia_sharpe_testes.csv` |
| Licença | MIT *(sugerida)* |

---

## Início — a pergunta que este notebook responde

O NB06 produziu o desempenho de 15 estratégias. Mas **diferença de Sharpe no ponto não é evidência**:
duas estratégias podem diferir por acaso amostral. Este notebook decide, com teste estatístico, se as
diferenças são **significativas** — é o que transforma "o BL teve Sharpe maior" em "o BL teve Sharpe
significativamente maior que o 1/N".

**O ponto metodológico central (e a coerência com o trabalho):** o teste clássico de diferença de Sharpe
— Jobson-Korkie (1981), corrigido por Memmel (2003) — **assume retornos normais e i.i.d.** Mas o próprio
trabalho documenta que os retornos **não são normais** (é a justificativa da PMPT). Usar só esse teste
seria contradição interna. Por isso aplica-se **também** o teste de **Ledoit-Wolf (2008)**, baseado em
*bootstrap* estacionário, **robusto à não-normalidade e à autocorrelação**. Os diagnósticos da Seção 3
(que mostram a não-normalidade das próprias séries de estratégia) **justificam empiricamente** essa
escolha — o mesmo fato que invalida o teste paramétrico é o que se mede antes de escolher o robusto.

**Escopo:** este NB7 trata da **inferência de desempenho das 15 estratégias**. Os diagnósticos
econométricos dos *retornos dos ativos* (que fundamentam a PMPT — ARCH/GARCH em 118 ativos, etc.)
permanecem válidos do trabalho anterior, pois os ativos não mudaram.

**Decisões (marcadas `DECISÃO`):**
- **Benchmark da comparação:** `EqualWeight` (1/N) — o desafiante de DeMiguel, Garlappi & Uppal (2009):
  "a sofisticação supera o 1/N?". `IBOV` também reportado.
- **Correção de múltiplas comparações:** Holm-Bonferroni (são 14 comparações simultâneas).
- **Bootstrap:** estacionário (Politis-Romano), bloco esperado ≈ 10 pregões, 2.000 reamostragens.

### Sumário
0. Dependências · 1. Parâmetros · 2. Insumos · 3. Diagnósticos por estratégia (justificam o bootstrap) ·
4. CAPM com erros HAC (alpha/beta) · 5. Testes de diferença de Sharpe (JK-Memmel + Ledoit-Wolf) ·
6. Diferença de Sortino (bootstrap) · 7. Exportação · 8. Fechamento

## 0. Dependências e reprodutibilidade (Regra 5)

In [ ]:
import sys
import numpy as np, pandas as pd, scipy, statsmodels, arch
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.multitest import multipletests
from arch.bootstrap import StationaryBootstrap
SEED=42; np.random.seed(SEED)
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scipy", scipy.__version__, "| statsmodels", statsmodels.__version__, "| arch", arch.__version__)

## 1. Parâmetros

In [ ]:
from pathlib import Path
parent_path = Path.cwd().parent.parent
DIR_ESTR    = parent_path / "data" / "Estrategias"
DIR_RET     = parent_path / "data" / "Retornos"
OUTPUT_DIR  = DIR_ESTR; OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRADING_DAYS = 252
BENCHMARK    = "EqualWeight"   # DECISÃO: desafiante 1/N (DeMiguel et al.). IBOV também reportado
ALPHA_SIG    = 0.05
HAC_MAXLAGS  = None            # None -> regra automática de Newey-West
BOOT_BLOCO   = 10              # DECISÃO: bloco esperado do bootstrap estacionário (pregões)
BOOT_REPS    = 2000            # DECISÃO: nº de reamostragens
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print(f"[OK] benchmark={BENCHMARK} | α={ALPHA_SIG} | bootstrap: bloco~{BOOT_BLOCO}, reps={BOOT_REPS}")

## 2. Insumos

Retornos diários das 15 estratégias + IBOV (NB06) e CDI (NB02).

In [ ]:
def _ler(d, nome, col=None):
    pq, csv = d/f"{nome}.parquet", d/f"{nome}.csv"
    df = (pd.read_parquet(pq) if pq.exists() else pd.read_csv(csv, index_col=0, parse_dates=True))
    df.index = pd.to_datetime(df.index); return df[col] if col else df

sr = _ler(DIR_ESTR, "strategy_returns")
rf = _ler(DIR_RET, "rf_diario", "cdi_diario").reindex(sr.index).ffill()
estrategias = [c for c in sr.columns if c != "IBOV"]
rf_periodo = float(rf.mean())
print(f"{len(estrategias)} estratégias × {sr.shape[0]} pregões "
      f"({sr.index.min().date()} → {sr.index.max().date()}) | benchmark: {BENCHMARK}")
assert BENCHMARK in estrategias, f"{BENCHMARK} não encontrado nas colunas"


## 3. Diagnósticos por estratégia — por que o bootstrap é necessário

Para cada série de retornos de estratégia: **Jarque-Bera** (normalidade), **Ljung-Box** (autocorrelação),
**ARCH-LM** (heterocedasticidade condicional) e **ADF** (estacionariedade). A expectativa — e o argumento
do trabalho — é que a normalidade seja **rejeitada** na maioria das estratégias, o que **invalida** o
teste paramétrico de Sharpe e justifica o de Ledoit-Wolf.

In [ ]:
def diagnosticos(r):
    r = r.dropna()
    jb, jbp, sk, ku = jarque_bera(r)
    lbp = acorr_ljungbox(r, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
    archp = het_arch(r, nlags=10)[1]
    adfp = adfuller(r, autolag="AIC")[1]
    return {"assimetria": sk, "curtose": ku, "JB_p": jbp, "LjungBox_p": lbp,
            "ARCH_p": archp, "ADF_p": adfp,
            "normal?": "não" if jbp < ALPHA_SIG else "sim"}

diag = pd.DataFrame({c: diagnosticos(sr[c]) for c in estrategias}).T
n_naonormal = (diag["normal?"] == "não").sum()
print(f"Normalidade rejeitada em {n_naonormal}/{len(estrategias)} estratégias "
      f"(JB p<{ALPHA_SIG}) -> teste paramétrico de Sharpe é mal-especificado; usar bootstrap.\n")
print(diag.round(4).to_string())

## 4. Modelo de mercado (CAPM) com erros-padrão HAC

Regressão do excesso de retorno de cada estratégia contra o excesso do IBOV, com erros-padrão
**Newey-West (HAC)**, robustos a heterocedasticidade e autocorrelação. Interessa o **alpha** (retorno
anormal — significativo?) e o **beta** (exposição ao mercado).

In [ ]:
def capm_hac(r_strat, r_mkt, rf_s, maxlags=HAC_MAXLAGS):
    df = pd.concat([r_strat, r_mkt, rf_s], axis=1).dropna()
    y = (df.iloc[:,0] - df.iloc[:,2]).values
    x = (df.iloc[:,1] - df.iloc[:,2]).values
    X = sm.add_constant(x)
    L = maxlags if maxlags is not None else int(np.floor(4*(len(y)/100)**(2/9)))
    m = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": L})
    return {"alpha_aa": m.params[0]*TRADING_DAYS, "alpha_p": m.pvalues[0],
            "beta": m.params[1], "beta_p": m.pvalues[1], "R2": m.rsquared}

capm = pd.DataFrame({c: capm_hac(sr[c], sr["IBOV"], rf) for c in estrategias}).T
print("CAPM (alpha anualizado, HAC Newey-West):\n")
print(capm.round(4).to_string())

## 5. Testes de diferença do Índice de Sharpe vs. benchmark

H0: $SR_{\text{estratégia}} = SR_{\text{benchmark}}$. Dois testes lado a lado:
- **Jobson-Korkie-Memmel (2003)** — paramétrico, assume normalidade (baseline);
- **Ledoit-Wolf (2008)** — *bootstrap* estacionário, robusto a não-normalidade/autocorrelação (válido).

P-valores ajustados por **Holm-Bonferroni** (14 comparações). O bootstrap é o teste a reportar como
principal; a divergência entre os dois é, ela mesma, informativa.

In [ ]:
def _sharpe(r, rf=0.0):
    r = np.asarray(r); s = r.std(ddof=1); return (r.mean()-rf)/s if s>0 else 0.0

def jk_memmel(ri, rj):
    ri=np.asarray(ri); rj=np.asarray(rj); T=len(ri)
    si, sj = _sharpe(ri), _sharpe(rj); rho=np.corrcoef(ri,rj)[0,1]
    theta=(1/T)*(2*(1-rho)+0.5*(si**2+sj**2-2*si*sj*rho**2))
    z=(si-sj)/np.sqrt(theta) if theta>0 else 0.0
    return 2*(1-stats.norm.cdf(abs(z)))

def lw_bootstrap_sharpe(ri, rj, bloco=BOOT_BLOCO, reps=BOOT_REPS, seed=SEED):
    data=np.column_stack([np.asarray(ri), np.asarray(rj)])
    diff_obs=_sharpe(data[:,0])-_sharpe(data[:,1])
    bs=StationaryBootstrap(bloco, data, seed=seed)
    diffs=np.array([_sharpe(p[0][0][:,0])-_sharpe(p[0][0][:,1]) for p in bs.bootstrap(reps)])
    se=diffs.std(ddof=1)
    z=diff_obs/se if se>0 else 0.0
    return diff_obs*np.sqrt(TRADING_DAYS), 2*(1-stats.norm.cdf(abs(z)))

base = sr[BENCHMARK].dropna()
linhas=[]
for c in estrategias:
    if c==BENCHMARK: continue
    d=pd.concat([sr[c], base], axis=1).dropna()
    p_jk = jk_memmel(d.iloc[:,0], d.iloc[:,1])
    dsr, p_lw = lw_bootstrap_sharpe(d.iloc[:,0], d.iloc[:,1])
    linhas.append({"estrategia": c, "dSR_anual_vs_bench": dsr, "JK_p": p_jk, "LW_p": p_lw})
testes=pd.DataFrame(linhas).set_index("estrategia")
testes["JK_p_holm"]=multipletests(testes["JK_p"], method="holm")[1]
testes["LW_p_holm"]=multipletests(testes["LW_p"], method="holm")[1]
testes["signif_LW_holm"]=np.where(testes["LW_p_holm"]<ALPHA_SIG, "sim", "não")
print(f"Diferença de Sharpe vs {BENCHMARK} (ΔSR anualizado; p ajustado por Holm):\n")
print(testes.round(4).to_string())

## 6. Diferença do Índice de Sortino (bootstrap)

Mesmo procedimento de *bootstrap* aplicado ao Sortino — coerente com a PMPT, por focar o *downside*.

In [ ]:
def _sortino(r, rf=0.0):
    r=np.asarray(r); exc=r-rf; dn=np.sqrt(np.mean(np.clip(exc,None,0)**2))
    return exc.mean()/dn if dn>0 else 0.0

def lw_bootstrap_sortino(ri, rj, rf=0.0, bloco=BOOT_BLOCO, reps=BOOT_REPS, seed=SEED):
    data=np.column_stack([np.asarray(ri), np.asarray(rj)])
    diff_obs=_sortino(data[:,0],rf)-_sortino(data[:,1],rf)
    bs=StationaryBootstrap(bloco, data, seed=seed)
    diffs=np.array([_sortino(p[0][0][:,0],rf)-_sortino(p[0][0][:,1],rf) for p in bs.bootstrap(reps)])
    se=diffs.std(ddof=1); z=diff_obs/se if se>0 else 0.0
    return diff_obs*np.sqrt(TRADING_DAYS), 2*(1-stats.norm.cdf(abs(z)))

linhas=[]
for c in estrategias:
    if c==BENCHMARK: continue
    d=pd.concat([sr[c], base], axis=1).dropna()
    dso, p = lw_bootstrap_sortino(d.iloc[:,0], d.iloc[:,1], rf_periodo)
    linhas.append({"estrategia": c, "dSortino_anual": dso, "LW_p": p})
sortino_test=pd.DataFrame(linhas).set_index("estrategia")
sortino_test["LW_p_holm"]=multipletests(sortino_test["LW_p"], method="holm")[1]
print(f"Diferença de Sortino vs {BENCHMARK} (bootstrap; p ajustado por Holm):\n")
print(sortino_test.round(4).to_string())

## 7. Exportação

In [ ]:
diag.to_csv(OUTPUT_DIR/"inferencia_diagnosticos.csv", float_format="%.6f")
capm.to_csv(OUTPUT_DIR/"inferencia_capm.csv", float_format="%.6f")
testes.to_csv(OUTPUT_DIR/"inferencia_sharpe_testes.csv", float_format="%.6f")
sortino_test.to_csv(OUTPUT_DIR/"inferencia_sortino_testes.csv", float_format="%.6f")
print("[OK] inferencia_diagnosticos.csv, inferencia_capm.csv, "
      "inferencia_sharpe_testes.csv, inferencia_sortino_testes.csv")

## 8. Fechamento — interpretação

*(Preencher após execução — Regra 1.)*

- **Alguma estratégia bate o 1/N de forma significativa?** Olhar `signif_LW_holm`. A literatura
  (DeMiguel et al., 2009) sugere que vencer o 1/N *com significância* é difícil — se nenhuma estratégia
  sofisticada rejeitar H0 após Holm, **isso é um resultado**, não um fracasso: corrobora a robustez do
  benchmark ingênuo em mercado emergente.
- **JK-Memmel × Ledoit-Wolf:** se o paramétrico acusa significância que o bootstrap não confirma, a
  diferença vinha da hipótese (falsa) de normalidade. Reportar os dois e privilegiar o bootstrap.
- **MPT × PMPT × BL:** as estratégias de *downside*/BL têm alpha (CAPM-HAC) significativo? Sortino
  significativamente superior?
- **BL clássico × downside:** a diferença de Sharpe/Sortino entre eles é significativa? Conecta ao
  comparativo de priors do NB6c (a inconsistência CAPM/PMPT também no desempenho).
- **Múltiplas comparações:** sempre citar o p **ajustado por Holm**, não o bruto — 14 testes inflam o
  erro tipo I.

**Limitações:** período único de backtest (sem subamostras); o bootstrap assume estacionariedade fraca,
tensionada pelas quebras estruturais documentadas; testes de Sharpe pressupõem que ele é a medida
relevante — daí complementar com Sortino.

---
# Autoavaliação — *Ten Simple Rules* (Rule et al., 2019)

| Regra | Tema | Status | Evidência / Ação |
| :---: | :--- | :---: | :--- |
| 1 | Contar uma história | 🟡 | Início/meio prontos; fim = seção 8, após execução |
| 2 | Documentar o processo | ✅ | Cabeçalho; decisões marcadas DECISÃO; escolha do teste justificada |
| 3 | Divisão clara de células | ✅ | Uma tarefa por célula; sumário |
| 4 | Modularizar código | 🟡 | Funções nomeadas; **ação:** `utils_pipeline.py` |
| 5 | Registrar dependências | 🟡 | Versões impressas; **ação:** `requirements.txt` (inclui statsmodels, arch) |
| 6 | Controle de versão | ⚪ | Git + `nbdime` |
| 7 | Construir um pipeline | ✅ | Consome NB06; saídas tabulares versionáveis |
| 8 | Compartilhar/explicar dados | 🟡 | Insumo derivado do NB06; `Datasets.md` |
| 9 | Ler, rodar e explorar | ✅ | Linear; sem *hidden state*; parâmetros no topo |
| 10 | Pesquisa aberta | 🟡 | Licença MIT + repositório sem dados |